# Optimization Mechanics (Learning Rates, Gradient Accumulation, and Loss Dynamics)

## Overview

Hyperparameter selection and execution scheduling in Supervised Fine-Tuning (SFT) directly dictate whether optimization converges smoothly or causes:

* Gradient exploding
* Catastrophic forgetting
* VRAM starvation

## Learning Rate Dynamics & Warmup Schedules

Unlike pre-training from scratch where initial learning rates are high ($\sim 10^{-3}$ to $10^{-4}$), fine-tuning starts from an already optimized parameter distribution $\theta_0$. Large gradient steps will destroy pre-trained features (catastrophic forgetting).

### Cosine Decay with Linear Warmup

The standard learning rate schedule for SFT is a linear warmup over $W$ steps followed by a cosine decay down to a minimum learning rate $\eta_{\min} = 0.1 \cdot \eta_{\max}$.

$$\eta(t) = \begin{cases} \eta_{\max} \cdot \frac{t}{W} & \text{if } t \le W \\ \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\pi \frac{t - W}{T - W}\right)\right) & \text{if } t > W \end{cases}$$

**Why Warmup is Non-Negotiable:**

* At step $0$, the task-specific classification head or LoRA adapters are initialized randomly or with zero weights
* Initial gradients $\nabla_\theta \mathcal{L}$ are high-variance
* Warmup prevents early gradient updates from destabilizing the base model's attention manifolds

### Typical Learning Rate Scale for SFT

* **Full Fine-Tuning:** $1 \times 10^{-6}$ to $2 \times 10^{-5}$
* **PEFT / LoRA** (Trainable parameters $\ll$ Total parameters): $1 \times 10^{-4}$ to $5 \times 10^{-4}$ (Higher rates are required to train low-rank matrices efficiently)

## Micro-Batching vs. Gradient Accumulation

To achieve stable optimization without hitting Out-Of-Memory (OOM) errors on fixed-memory GPUs, we decouple Physical Micro-Batch Size from Effective Global Batch Size.

### Mathematical Equivalence

Let $B_{\text{micro}}$ be the micro-batch size processing sequences of length $L$ per GPU across $D$ distributed devices. Let $K$ be the gradient accumulation steps. The Effective Global Batch Size $B_{\text{global}}$ is:

$$B_{\text{global}} = B_{\text{micro}} \times D \times K$$

Instead of updating parameters $\theta$ every forward-backward pass, gradients are accumulated into parameter gradient buffers $\nabla_\theta \mathcal{L}$:

$$\nabla_\theta \mathcal{L}_{\text{accum}} = \frac{1}{K} \sum_{k=1}^K \nabla_\theta \mathcal{L}(X_k, Y_k)$$

Only on step $k = K$ does the optimizer perform its parameter update step: $\theta \leftarrow \text{Optimizer}(\theta, \nabla_\theta \mathcal{L}_{\text{accum}})$ and zero out the gradient buffers (`optimizer.zero_grad()`).

### Systems VRAM Overhead

* **Micro-batch size** ($B_{\text{micro}}$) drives peak VRAM usage because activation memory scales linearly with $B_{\text{micro}}$
* **Gradient Accumulation** ($K$) costs zero extra VRAM for activation storage (activations are freed immediately during each backward pass) at the expense of running $K$ forward-backward passes per optimizer step

## Overfitting, Loss Signals, and Catastrophic Forgetting

Monitoring training metrics in SFT requires different diagnostics than standard pre-training.

### Diagnosing Train-vs-Validation Loss Divergence

```
Loss
 │
 │      /─── Validation Loss (Rising -> Overfitting / Memorization)
 │     /
 │    /───── Training Loss (Decreasing)
 │   /
 └──┴──────────────────────────────► Step / Epochs
```

### SFT Overfitting Pattern

Unlike pre-training, where training and validation loss decay together, SFT on small datasets ($\le 10,000$ samples) quickly causes validation loss to plateau or increase while training loss continues to drop toward zero.

**Root Cause:** The model transitions from learning generalized task formats to memorizing specific exact string sequences from the training set.

### Mitigation Strategies

* **Early Stopping:** Halt training when validation loss stops improving for $N$ evaluation checkpoints
* **Dataset Regularization:** Inject general-domain instruction datasets (e.g., 10-20% SlimPajama or general OpenHermes instructions) alongside domain data to anchor general capabilities
* **Weight Decay:** Set AdamW weight decay $\lambda \in [0.01, 0.1]$ to penalize large weight shifts in trainable parameters

---

**Module 1 Complete:** This completes the theoretical and systems foundation for Module 1 (Fine-Tuning Fundamentals).